# Part 1 — AI on NRP

A subset of NRP's GPUs serve open-weight language models behind one **OpenAI-compatible endpoint**, `https://ellm.nrp-nautilus.io/v1`. No pod to run, no GPU to hold, no per-seat license.

This notebook shows two things you get on a hub like this one:

1. **An LLM you can call from Python**, with the token already in your environment.
2. **Jupyter AI**, an assistant inside JupyterLab that can explain and fix code, and look at your data.

> ⚠️ **NRP is not HIPAA compliant.** Treat every prompt as non-confidential: public, synthetic and de-identified data only. No PHI.

> 📘 [Managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/) · [Available models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) · [Get your own token](https://nrp.ai/llmtoken)

## 1. The token is already here

On this training hub, `OPENAI_API_BASE` and `OPENAI_API_KEY` are exported into every server, so **nobody handles a key**. Elsewhere you mint one at [nrp.ai/llmtoken](https://nrp.ai/llmtoken) and set the same two variables.

In [ ]:
import os

print("base:", os.environ.get("OPENAI_API_BASE", "(not set)"))
key = os.environ.get("OPENAI_API_KEY", "")
print("key: ", (key[:8] + "…") if key else "(not set)")

## 2. What models are on offer

One endpoint fronts the whole catalog: large and small chat models, coding models, vision and audio models, and an embedding model.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=os.environ["OPENAI_API_BASE"])

for m in client.models.list().data:
    print(m.id)

The list also contains task aliases such as `coding` or `vision`, which point at whichever model currently suits that task. Models come and go as better open models appear, so check the [models page](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) before pinning a name into a pipeline.

> **The model list doesn't check your token.** It answers without authentication. The first call that really tests the token is a chat completion, the next cell.

## 3. Ask it something

The standard `openai` package talks to NRP unchanged; only `base_url` differs from the commercial API. `gpt-oss` is a fast, general-purpose model.

One wrinkle: the SDK looks for `OPENAI_BASE_URL`, but NRP exports `OPENAI_API_BASE`, so pass it explicitly (as above) or the client quietly calls `api.openai.com` instead.

In [ ]:
resp = client.chat.completions.create(
    model="gpt-oss",
    messages=[
        {"role": "user",
         "content": "In two sentences, what is the National Research Platform?"},
    ],
)

print(resp.choices[0].message.content)

**That is the whole trick.** The same lines work against a commercial API, against NRP, or against a vLLM server you run yourself on a GPU: you change `base_url` and nothing else.

## 4. Jupyter AI: the chat panel

The hub ships [Jupyter AI](https://jupyter-ai.readthedocs.io/) already pointed at NRP. Click the **chat icon** in the left sidebar and ask it something.

> **The chat panel can't see your notebook**: not the kernel, your variables or your outputs. Section 5 shows the way around that.

### Break something on purpose

The next three cells are broken. Run them, then select one and ask the chat panel what went wrong, or type `/fix` with a crashed cell selected. The first two crash. The third doesn't, and **that's the interesting one**: it runs, prints a number, and the number is wrong.

In [ ]:
# 🐞 BROKEN — temperature conversion   (this one crashes)
temperatures_c = [18, 21, 25, 30, 12]

def to_fahrenheit(c):
    return c * 9 / 5 + 32

for t in temperatures_c:
    print(t, "C =", to_farenheit(t), "F")

In [ ]:
# 🐞 BROKEN — class average   (this one crashes)
student_scores = {"ana": 88, "ben": 92, "cleo": 79}

total = 0
for name in student_scores:
    total += name

print("class average:", total / len(student_scores))

In [ ]:
# 🐞 BROKEN — sensor average   (no crash; the answer is just wrong)
readings = [3, 7, 2, 9, 4, 8]

# intended: the average of every reading after the first
average = sum(readings[1:]) / len(readings)

print("average of readings 2..6 =", average)
print("expected:", (7 + 2 + 9 + 4 + 8) / 5)

> **`/fix` only works on cells that crashed.** It needs a selected cell *with error output*. The sensor cell has no traceback, so the assistant has nothing to react to: you have to tell it what the code was meant to do. That's the bug that survives into a paper's results.

## 5. The `%%ai` magics: an assistant that sees your data

A **magic** is a notebook command that isn't Python. Jupyter AI's magics can reach into the kernel, which the chat panel can't.

In [ ]:
%load_ext jupyter_ai_magics

The model name after `openai-chat:` goes straight to NRP, so use any name from section 2. (`%ai list` prints Jupyter AI's own built-in catalog, which is **not** NRP's; ignore it.)

In [ ]:
%%ai openai-chat:gpt-oss
What is the National Research Platform, in two sentences?

### Give it your data

Anything in `{curly braces}` inside a `%%ai` prompt is replaced by the **live value** of that variable. So you can hand the model a summary of a real table and ask about it.

The data here is scikit-learn's bundled **diabetes dataset**: public, de-identified, 442 patients, ten baseline measurements and a disease-progression score one year later.

In [ ]:
import pandas as pd

try:
    from sklearn.datasets import load_diabetes
    df = load_diabetes(as_frame=True, scaled=False).frame
except ImportError:  # no scikit-learn: a small synthetic stand-in
    import numpy as np
    rng = np.random.default_rng(0)
    df = pd.DataFrame({"age": rng.integers(20, 80, 200), "bmi": rng.normal(26, 4, 200).round(1),
                       "bp": rng.normal(95, 14, 200).round(), "target": rng.integers(25, 346, 200)})

summary = df.describe().round(1).to_string()
print(summary)

In [ ]:
%%ai openai-chat:gpt-oss
Here is a summary of a public, de-identified dataset (one row per patient; "target" is
disease progression one year later):

{summary}

Which three columns would you look at first to predict the target, and what would you
check before trusting this data? Keep it short.

The model received the real table, not a description of it. That's the pattern for "help me understand my data" questions, as long as the data is allowed to leave your machine.

**Check the answer.** The model's suggestions are a starting point, not a result. One line of pandas shows how each column actually relates to the target, including the sign:

In [ ]:
df.corr()["target"].drop("target").sort_values(key=abs, ascending=False).round(2)

### Explain the last error

`%ai error` sends the most recent traceback to the model, with nothing to copy or paste. The hook is installed by `%load_ext`, so **re-run one of the crashing cells above first**.

In [ ]:
%ai error openai-chat:gpt-oss

### The silent bug, with live values

Re-run the sensor cell here so `readings` and `average` are live (and still wrong), then ask about them.

In [ ]:
# 🐞 BROKEN — sensor average, again   (re-run so the wrong values are live below)
readings = [3, 7, 2, 9, 4, 8]

# intended: the average of every reading after the first
average = sum(readings[1:]) / len(readings)

print("average of readings 2..6 =", average)
print("expected:", (7 + 2 + 9 + 4 + 8) / 5)

In [ ]:
%%ai openai-chat:gpt-oss
Here are some sensor readings: {readings}

My code computed an average of {average}, but what I wanted was the average of
every reading *after the first one*. Is {average} the right answer? Show the
arithmetic either way.

## What to take away

- The endpoint is **OpenAI-compatible**, so notebooks, Jupyter AI, agents and your own scripts all work against NRP with a `base_url` change.
- The chat panel can't see your kernel; the `%%ai` magics can, through `{variables}`.
- The assistant proposes; you still read the proposal and choose. Asked to fix a cell, it often offers more than one correct answer.

Next: [Part 2 — GPUs on NRP](2_gpu.ipynb).